In [6]:
import pandas as pd

df = pd.read_csv("../../adult_group_02(in).csv")  # use the path your team agreed on

print(df.shape) # rows, columns
print(df.dtypes) # Python types
print(df.isna().sum()) # missing values per column
print(df.nunique()) # unique values per column
print((df["age"] < 0).sum()) # impossible ages
print(df["gender"].value_counts(dropna=False)) # typos + missing
print((df["marital-status"] == "?").sum()) # hidden missing
print(pd.crosstab(df["tax_category"], df["income"])) # proves leakage
print(pd.crosstab(df["workclass"], df["workclass_id"])) # proves duplicate



(40190, 14)
age                 int64
workclass             str
education             str
marital-status        str
occupation            str
race                  str
gender                str
capital-gain        int64
capital-loss        int64
hours-per-week    float64
income                str
workclass_id        int64
tax_category          str
survey_program        str
dtype: object
age                  0
workclass            0
education            0
marital-status       0
occupation           0
race                 0
gender            1206
capital-gain         0
capital-loss         0
hours-per-week    1607
income               0
workclass_id         0
tax_category         0
survey_program       0
dtype: int64
age                76
workclass           8
education          16
marital-status      8
occupation         14
race                5
gender              6
capital-gain      123
capital-loss       99
hours-per-week     95
income              2
workclass_id        8
tax_categor

**Data dictionary** — 40,190 people, 14 columns.

| Column | Meaning | Type | Notes |
|---|---|---|---|
| age | Age of the person | Number | Ages go from 17 to 90. 201 ages are wrong (−10, −5), so we treat them as missing. |
| workclass | Who the person works for (private, government, …) | Category | 8 groups. "Without-pay" and "Never-worked" are very rare. |
| education | Highest education level | Category with order | 16 levels, from Preschool to Doctorate. |
| marital-status | Married, divorced, single, … | Category | 1,206 values are "?", which means missing. |
| occupation | Type of job | Category | 14 jobs. Nothing missing. |
| race | Race of the person | Category | 5 groups. Most people (83%) are White. Sensitive data, so be careful. |
| gender | Male or female | Two values | Has typos ("Mle", "Femle", "male") and 1,206 missing. Sensitive data, so be careful. |
| capital-gain | Money earned from investments | Number | Most people have 0. 242 people have 99,999, which is a maximum value, not a real amount. |
| capital-loss | Money lost on investments | Number | Most people have 0. |
| hours-per-week | Hours worked per week | Number | 1,607 missing. Goes from 1 to 99. |
| income | Earns more than \$50K a year, or not | Two values — **what we predict** | 74% earn ≤50K, 26% earn >50K. |
| workclass_id | Number code for workclass | Category (stored as a number) | Exact copy of workclass, so we remove it. |
| tax_category | Tax group: lower or higher | Two values | Always matches income, so it gives away the answer. We remove it. |
| survey_program | Name of the survey | Same value everywhere | Always "Census", so it tells us nothing. We remove it. |

## 0.1 Problem & Evaluation Design

### Alternative targets

| Other thing to predict | What we predict | Why useful |
|---|---|---|
| hours-per-week | How many hours a person works each week (a number) | Companies can plan their staff better |
| occupation | What kind of job a person has (one of 14 groups) | Job services can suggest jobs that fit the person |

### Final problem definition

**Business problem:** A bank has a new expensive credit card for people who earn a lot. The bank does not know how much people earn. Sending an offer costs money, so the bank cannot send it to everyone.

**Decision to support:** Do we send this person the offer? Yes or no.

**Unit of observation:** One row is one person.

**Prediction moment:** When the bank knows the person's basic info (age, job, education, hours worked, …), before sending offers.

**Success:** The model finds most of the people who earn more than 50K, and does not pick too many people who earn less. It must do better than the simple rule and the Dummy model.

**Target:** `income` — does the person earn more than 50K dollars a year? (yes = >50K, no = ≤50K). About 1 in 4 people earn more than 50K.

**Task:** Classification (sort people into two groups)

### Rule-based baseline

**Rule:** If the person is married → guess "earns more than 50K". If not → guess "earns 50K or less".

**Why this is useful:** Married people earn more than 50K much more often (46%) than people who never married (6%). The rule uses only one clue. Our ML models use all clues together (job, education, hours, …), so they should do better. If they don't, the ML is not worth it.

### Evaluation metrics

**Main score:** F1 score for the "more than 50K" group.

**Why:** Only 1 in 4 people earn more than 50K. A lazy model that always says "50K or less" is right 74% of the time, but finds nobody rich. So "percentage correct" (accuracy) is a bad score here.

F1 checks two things at once:
- Did we find most of the rich people? (**recall**)
- Were the people we picked really rich? (**precision**)

For the bank, missing a rich customer costs more than one wasted letter. But sending letters to everyone costs too much. F1 looks for a good balance.

**Extra scores:** Recall and precision (to see which mistake happens more), ROC-AUC, and accuracy (only to show why accuracy fools us here).